## 0. Setup (run first, every session) - and how your progress is protected
**Colab:** checkpoints/results are written *directly to your Google Drive* (symlinks), so a power or internet cut loses at most one checkpoint interval (~300 updates). After reconnecting: re-run section 0, then re-run the same training cell - it resumes automatically. Drive's free quota is 15 GB: one full-FT run is ~12 GB, so use the cleanup cell after each full-FT run.

**Kaggle:** an interactive session's disk is wiped when it ends. Protect yourself with **Save Version -> Save & Run All (Commit)**: it runs on Kaggle's servers even if your internet/power dies, and the output (checkpoints + results, up to 20 GB) is kept. To continue later: in the new notebook use *Add Input -> Notebook Output Files* and pick the previous version; section 0 copies its checkpoints back and training resumes. Do one method per commit run (must finish within the 9 h limit). Settings: Accelerator = GPU, Internet = On.

4. Copy the assignment's `first_plot_questions.yaml` to `PA 3/configs/first_plot_questions.yaml` locally and commit it (it is small), so the notebook gets it from GitHub.
5. Budget (estimates, not measured): one full-FT run is a few hours on a T4; LoRA runs are faster. Run **one method per session/commit**. One full-FT run is ~12 GB (Kaggle limit 20 GB, Drive 15 GB), so delete its `recovery_*.pt` once it has finished (cell provided).

In [ ]:
import os, sys, subprocess, shutil, glob
REPO_URL = "https://github.com/ZainFarhan-Work/ATML.git"
BRANCH   = "main"
ON_KAGGLE = os.path.exists("/kaggle")
ROOT = "/kaggle/working" if ON_KAGGLE else "/content"

if not os.path.exists(f"{ROOT}/ATML"):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, f"{ROOT}/ATML"], check=True)
else:
    subprocess.run(["git", "-C", f"{ROOT}/ATML", "pull"], check=False)
os.chdir(f"{ROOT}/ATML/PA 3")
print(os.getcwd())

PERSIST = ["checkpoints", "results", "cached", "splits"]
if not ON_KAGGLE:
    # Colab: keep run state on Drive so it survives disconnects
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive/atml_pa3"
    for d in PERSIST:
        os.makedirs(f"{DRIVE}/{d}", exist_ok=True)
        if os.path.isdir(d) and not os.path.islink(d):
            shutil.copytree(d, f"{DRIVE}/{d}", dirs_exist_ok=True); shutil.rmtree(d)
        if not os.path.islink(d):
            os.symlink(f"{DRIVE}/{d}", d)
else:
    # Kaggle: restore from earlier committed runs attached as inputs
    for src_root in glob.glob("/kaggle/input/*/ATML/PA 3") + glob.glob("/kaggle/input/*/*/ATML/PA 3"):
        for d in PERSIST:
            if os.path.isdir(f"{src_root}/{d}"):
                shutil.copytree(f"{src_root}/{d}", d, dirs_exist_ok=True)
                print("restored", d, "from", src_root)

def sync_now():
    # Colab: nothing to do (already on Drive). Kaggle: output is saved when the committed run finishes.
    print("progress is stored in:", "Google Drive" if not ON_KAGGLE else "/kaggle/working (kept by Save & Run All)")

In [ ]:
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -m scripts.check_environment

## 1.0 Smoke test (5 updates, ~minutes) - do this before spending real compute
Reports seconds/update and peak GPU memory for full FT. If it runs out of memory, set `micro_batch: 1` and `grad_accum: 32` in `configs/task1_llm.yaml` (same effective batch of 32).

In [ ]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m task1_llm.train --method full_ft --max-updates 5
import json; print(json.load(open("results/task1/full_ft/resources.json")))
# remove the smoke-test artifacts so the real run starts clean
!rm -rf results/task1/full_ft checkpoints/task1/full_ft

## 1.1 Full fine-tuning (lr 2e-5, lambda 0)
Resumable: if the session dies, re-run section 0 and this cell.

In [ ]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m task1_llm.train --method full_ft --resume
sync_now()

## 1.2 Full FT + L2-SP (lambda 1e-3)
Starts again from the original checkpoint. theta_0 is kept on CPU (`ref_device: cpu`) to fit 16 GB.

In [ ]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m task1_llm.train --method full_ft_l2sp --resume
sync_now()

### Package the Full-FT snapshots for download (do this in the Kaggle session that holds them)
Creates one small `bundle_<update>.zip` per snapshot (~1.2 GB: FP16 model for evaluation + the FP32 `down_proj` weights for the diagnostics). Download these files from the file panel (`checkpoints/task1/<method>/bundle_*.zip`) together with `results/task1/`. Skip any snapshot you already evaluated/packaged.

In [ ]:
!python -m task1_llm.extract_snapshot --method full_ft --update 625
!python -m task1_llm.extract_snapshot --method full_ft --update 1250
!python -m task1_llm.extract_snapshot --method full_ft_l2sp --update 1250
!ls -lh checkpoints/task1/*/bundle_*.zip

### Free space once a full-FT run has FINISHED
Only the eval snapshots (`eval_625`, `eval_1250`) are needed afterwards; the recovery checkpoint is 7 GB.

In [ ]:
!rm -f checkpoints/task1/full_ft/recovery_*.pt checkpoints/task1/full_ft_l2sp/recovery_*.pt
!df -h . | tail -1

## 1.3 LoRA rank 4 and 16 (lr 2e-4)

In [ ]:
for m in ["lora4", "lora16"]:
    !PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m task1_llm.train --method {m} --resume
    sync_now()

## 1.4 OPLoRA (r=16, k=128)
Computes FP32 SVD bases once (time/storage logged), then trains with effective factors.

In [ ]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m task1_llm.train --method oplora16 --resume
sync_now()

## 1.5 Evaluation of the seven conditions (GSM8K, HellaSwag, BoolQ, ARC-Easy)
Slow part is GSM8K generation (1,319 questions, up to 1,024 tokens). Finished conditions are skipped on re-run, so this is safe to resume. Use `--only baseline full_ft_mid` etc. to split across sessions.

In [ ]:
!python -m task1_llm.evaluate
sync_now()

## 1.6 Diagnostics, qualitative generations, tables and figures

In [ ]:
!python -m task1_llm.diagnostics
!python -m task1_llm.qualitative
!python -m task1_llm.analyze
sync_now()

## Final: download what you need locally
Colab: copy `results/` (small) from Drive `atml_pa3/results` to `PA 3/results` locally. Kaggle: download the committed version's output (results folder; checkpoints only if needed).